# Classifier Validation — Stratified Human Review

Validates the GPT-4o Mini classifier against human judgment. Produces inter-rater reliability (IRR) and classifier-vs-human agreement statistics.

**Design:** `TOTAL_N` responses stratified across 5 risk levels (`STRATA_N` per level). Human raters independently code each response as `1` (direct) or `0` (non-direct); disagreements resolved by tiebreaker.

**Inputs required:**
- `../joined_dataset.csv` — full corpus with classifier labels
- `rater_sheet_completed.csv` — completed human ratings (sections 3–4)

**Sections:**
1. Sample responses (stratified by risk level)
2. Export rater sheet
3. Load completed ratings and resolve disagreements
4. Inter-rater reliability and classifier agreement

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

VALIDATION_DIR = Path(".")
# Set this to your joined dataset path, e.g. the published paper_data/ file
# or a file you generated with scripts/aggregate.py
DATA_FILE      = Path("../paper_data/joined_dataset_redacted.csv")

RISK_LEVELS = ['very_low', 'low', 'medium', 'high', 'very_high']
STRATA_N    = 60    # responses per risk level
TOTAL_N     = STRATA_N * len(RISK_LEVELS)  # 300
SEED        = 42

VALIDATION_DIR = Path(".")
VALIDATION_DIR = Path(".")
VALIDATION_DIR = Path(".")


## 1. Sample responses (stratified by risk level)

In [2]:
corpus = pd.read_csv(DATA_FILE)
corpus_api = corpus[corpus['access_method'] == 'api']

print(f'API corpus: {len(corpus_api):,} rows')
print(corpus_api['risk_category'].value_counts()[RISK_LEVELS])

API corpus: 15,000 rows
risk_category
very_low     2000
low          2500
medium       5000
high         2500
very_high    3000
Name: count, dtype: int64


In [3]:
rng  = np.random.default_rng(SEED)
rows = []

for risk in RISK_LEVELS:
    pool = corpus_api[corpus_api['risk_category'] == risk]
    rows.append(pool.sample(n=STRATA_N, random_state=rng.integers(1_000_000)))
    print(f'  {risk:10s}: sampled {STRATA_N} from {len(pool):,}')

keep_cols = ['model_id', 'prompt_id', 'prompt_text', 'risk_category',
             'response_text', 'is_direct', 'iteration', 'run_id']

gold = pd.concat(rows, ignore_index=True)[keep_cols]
gold.insert(0, 'sample_id', range(1, len(gold) + 1))

print(f'Total sample: {len(gold)}')
print(gold['risk_category'].value_counts()[RISK_LEVELS])

gold.to_csv(GOLD_FILE, index=False, encoding='utf-8-sig')
print(f'Saved: {GOLD_FILE}')

  very_low  : sampled 60 from 2,000
  low       : sampled 60 from 2,500
  medium    : sampled 60 from 5,000
  high      : sampled 60 from 2,500
  very_high : sampled 60 from 3,000
Total sample: 300
risk_category
very_low     60
low          60
medium       60
high         60
very_high    60
Name: count, dtype: int64
Saved: sample_gold.csv


## 2. Export rater sheet

In [ ]:
rater_sheet = gold[['sample_id', 'prompt_text', 'response_text']].copy()
rater_sheet['rater1_code'] = pd.NA
rater_sheet['rater2_code'] = pd.NA
rater_sheet['notes']       = pd.NA

rater_sheet = rater_sheet.sample(frac=1, random_state=SEED).reset_index(drop=True)

rater_sheet.to_csv(RATER_FILE, index=False, encoding='utf-8-sig')
print(f'Saved {RATER_FILE} ({len(rater_sheet)} rows)')
print('Fill rater1_code and rater2_code independently (1=direct, 0=non-direct).')
print('Add tiebreaker where raters disagree, then save as rater_sheet_completed.csv')

## 3. Load completed ratings and resolve disagreements

In [7]:
if not COMPLETED_FILE.exists():
    print(f'{COMPLETED_FILE.name} not yet available — complete the rater sheet and save it as rater_sheet_completed.csv')
else:
    gold      = pd.read_csv(GOLD_FILE, encoding='utf-8-sig')
    completed = pd.read_csv(COMPLETED_FILE, encoding='utf-8-sig')

    rated = gold.merge(
        completed[['sample_id', 'rater1_code', 'rater2_code', 'tiebreaker']],
        on='sample_id', how='inner'
    )

    rated['agree'] = rated['rater1_code'] == rated['rater2_code']

    def resolve(row):
        if row['agree']:
            return int(row['rater1_code'])
        elif pd.notna(row.get('tiebreaker')):
            return int(row['tiebreaker'])
        return pd.NA

    rated['human_label'] = rated.apply(resolve, axis=1)

    n_agree      = rated['agree'].sum()
    n_unresolved = rated['human_label'].isna().sum()
    print(f'Agreement:     {n_agree}/{len(rated)} ({100*n_agree/len(rated):.1f}%)')
    print(f'Disagreements: {(~rated["agree"]).sum()}')
    if n_unresolved > 0:
        print(f'WARNING: {n_unresolved} unresolved disagreements (no tiebreaker set)')

    rated.to_csv(VALIDATION_DIR / 'rated.csv', index=False, encoding='utf-8-sig')
    print('Saved rated.csv')

Agreement:     292/300 (97.3%)
Disagreements: 8
Saved rated.csv


## 4. Inter-rater reliability and classifier agreement

In [8]:
if not COMPLETED_FILE.exists():
    print('Skip — rater_sheet_completed.csv not yet available')
else:
    rated_clean = rated[rated['human_label'].notna()].copy()
    rated_clean['human_label']      = rated_clean['human_label'].astype(int)
    rated_clean['classifier_label'] = rated_clean['is_direct'].astype(int)

    # Inter-rater reliability
    irr_df    = rated_clean.dropna(subset=['rater1_code', 'rater2_code'])
    irr_kappa = cohen_kappa_score(irr_df['rater1_code'].astype(int), irr_df['rater2_code'].astype(int))
    irr_pct   = (irr_df['rater1_code'] == irr_df['rater2_code']).mean()
    print('=== Inter-rater reliability (rater1 vs rater2) ===')
    print(f'  n={len(irr_df)}, % agreement={irr_pct:.3f}, Cohen κ={irr_kappa:.3f}')

    # Classifier vs human — overall
    clf_kappa = cohen_kappa_score(rated_clean['human_label'], rated_clean['classifier_label'])
    clf_pct   = (rated_clean['human_label'] == rated_clean['classifier_label']).mean()
    print('\n=== Classifier vs human (overall) ===')
    print(f'  n={len(rated_clean)}, % agreement={clf_pct:.3f}, Cohen κ={clf_kappa:.3f}')

=== Inter-rater reliability (rater1 vs rater2) ===
  n=300, % agreement=0.973, Cohen κ=0.920

=== Classifier vs human (overall) ===
  n=300, % agreement=0.963, Cohen κ=0.893


In [9]:
if not COMPLETED_FILE.exists():
    print('Skip — rater_sheet_completed.csv not yet available')
else:
    rows = []
    for risk in RISK_LEVELS:
        sub = rated_clean[rated_clean['risk_category'] == risk]
        if len(sub) < 2:
            continue
        try:
            k = cohen_kappa_score(sub['human_label'], sub['classifier_label'])
        except Exception:
            k = float('nan')
        pct = (sub['human_label'] == sub['classifier_label']).mean()
        rows.append({'risk_category': risk, 'n': len(sub),
                     'pct_agreement': round(pct, 3), 'kappa': round(k, 3)})

    strata_results = pd.DataFrame(rows)
    print('=== Classifier agreement by risk level ===')
    print(strata_results.to_string(index=False))
    strata_results.to_csv(VALIDATION_DIR / 'strata_agreement.csv', index=False)

    print('\n=== Summary ===')
    print(f'Validation sample: n={len(rated_clean)}')
    print(f'Strata: {len(RISK_LEVELS)} risk levels ({STRATA_N} per level)')
    print(f'Inter-rater reliability: % agreement={irr_pct:.0%}, κ={irr_kappa:.2f}')
    print(f'Classifier vs human:     % agreement={clf_pct:.0%}, κ={clf_kappa:.2f}')

=== Classifier agreement by risk level ===
risk_category  n  pct_agreement  kappa
     very_low 60          0.983  0.000
          low 60          1.000    NaN
       medium 60          0.983  0.946
         high 60          0.900  0.571
    very_high 60          0.950  0.864

=== Summary ===
Validation sample: n=300
Strata: 5 risk levels (60 per level)
Inter-rater reliability: % agreement=97%, κ=0.92
Classifier vs human:     % agreement=96%, κ=0.89


/Users/ellice/Desktop/llm-evals/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/Users/ellice/Desktop/llm-evals/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar divide
  k = xp.sum(w_mat * confusion) / xp.sum(w_mat * expected)
